In [1]:
pip install groq

Note: you may need to restart the kernel to use updated packages.


In [3]:
from groq import Groq

client = Groq(api_key = "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")
completion = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[
      {
        "role": "system",
        "content": "You are a Enterprise Help line."
      },
      {
           "role": "user",
        "content": "Casual Leave"
      }
    ]
)
print(completion.choices[0].message.content)

**Hi there!**  
It looks like you’d like to take some casual leave. I’m happy to walk you through the typical process and answer any questions you might have.

---

## 1️⃣ What is “Casual Leave”?

* **Purpose:** Short‑term, unscheduled time off for personal matters (e.g., appointments, family events, urgent errands).  
* **Eligibility:** Usually available to all full‑time employees after a probation period; part‑time staff may receive a pro‑rated amount.  
* **Accrual:** Most companies grant **X–Y days per calendar year** (e.g., 6–12 days) that roll over only if your policy allows it.  

> **Tip:** Check your employee handbook or the HR portal for the exact number of days you’re entitled to.

---

## 2️⃣ How to Apply for Casual Leave

| Step | Action | Typical System/Tool |
|------|--------|----------------------|
| **1** | **Check your balance** – Make sure you have enough casual‑leave days left. | HRIS/People portal (e.g., Workday, SAP SuccessFactors) |
| **2** | **Notify your manage

In [4]:
!pip install -q langchain langchain-community langchain-groq langchain-huggingface langchain-chroma sentence-transformers chromadb pymupdf

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
streamlit 1.51.0 requires protobuf<7,>=3.20, but you have protobuf 7.36.1 which is incompatible.


In [5]:
pip install "protobuf<7,>=3.20"


  Attempting uninstall: protobuf
    Found existing installation: protobuf 7.36.1
    Uninstalling protobuf-7.36.1:
      Successfully uninstalled protobuf-7.36.1
Note: you may need to restart the kernel to use updated packages.


In [1]:
import pymupdf
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_text_splitters import RecursiveCharacterTextSplitter
import torch
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_groq import ChatGroq

In [ ]:
pdf_path = "Northgate_Solutions_HR_Policy_Manual.pdf"
embed_model = "sentence-transformers/all-MiniLM-L6-v2"  # Hugging face
chunk_size =500
chunk_overlap = 50
top_k =5
device = "cpu"

In [ ]:
doc =pymupdf.open("Northgate_Solutions_HR_Policy_Manual.pdf")   ##Opening the pdf
pages =[]                                                         #Store each document 
for i, page in enumerate (doc):
  result = Document(page_content=page.get_text(),
                    metadata = {"page":i, "source":"Northgate_Solutions_HR_Policy_Manual.pdf"})  #Store the data into each page with index numbers
  pages.append(result)

In [9]:
pages[5]

Document(metadata={'page': 5, 'source': 'Northgate_Solutions_HR_Policy_Manual.pdf'}, page_content='•\nCompany laptops and accounts must not be used to access unauthorized or non-business-related\nservices that pose a security risk.\n•\nMulti-factor authentication (MFA) is mandatory for all company systems, including email and internal\ntools.\n•\nClient data must never be stored on personal devices or personal cloud storage accounts.\n9.2 Incident Reporting\nAny suspected data breach, phishing attempt, or lost/stolen device must be reported to the IT Security team\nwithin 2 hours of discovery via the emergency IT hotline.\n10. Separation & Exit Policy\n10.1 Notice Period\nThe standard notice period is 60 days for employees at Associate level and above, and 30 days for\nTrainee/Intern roles. The company reserves the right to relieve an employee earlier with pay in lieu of the\nremaining notice period.\n10.2 Exit Process\n•\nExit interview conducted by HR within the final week of the not

In [10]:
chunks=RecursiveCharacterTextSplitter(chunk_size = chunk_size,
                                      chunk_overlap = chunk_overlap,
                                      separators=["\n\n", "\n", " ", "."]
).split_documents(pages)

print(f"The page length {len(pages)} -> chunks will be {len(chunks)}")

The page length 6 -> chunks will be 22


In [11]:
embeddings = HuggingFaceEmbeddings(model_name = embed_model)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [12]:
vectorstore = Chroma.from_documents(chunks, embedding= embeddings)

In [13]:
llm = ChatGroq(
    model="openai/gpt-oss-120b",
    api_key="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"
)

In [20]:
# prompt
prompt = ChatPromptTemplate.from_template(
    """
    You are a helpful HR Assistant.

    Answer the question using ONLY the information provided in the context below.

    If the question cannot be answered from the provided context, or if the
    relevant information is not available in the context, do not guess or
    make up an answer.

    Instead, respond exactly with:
    "I couldn't find this information in the available HR documents.
    Please contact HR for clarification."

    Do not use outside knowledge for company-specific HR questions.

    <context>
    {context}
    </context>

    <question>
    {question}
    </question>

    Answer:
    """
)

In [21]:
retriever= vectorstore.as_retriever(search_kwargs ={"k":top_k})

In [22]:
#Ragchain
answer_chain = prompt | llm | StrOutputParser()

In [23]:
question = "what is adress of canditate"
context = retriever.invoke(question)
answer = answer_chain.invoke({"context":context, "question": question})
print(answer)

I couldn't find this information in the available HR documents.
Please contact HR for clarification.


In [25]:
question = "What leave policy"
context = retriever.invoke(question)
answer = answer_chain.invoke({"context":context, "question": question})
print(answer)

Based on the HR policy manual:

- **Casual Leave (CL)** – Employees are entitled to **12 days of casual leave per calendar year**, which are credited on a monthly basis.  
- **Unpaid Leave** – If an employee has exhausted their paid‑leave balance, they may request **up to 30 days of unpaid leave per year**, subject to approval by their department head and HR Business Partner.
